## CS310 Natural Language Processing
## Assignment 1. Neural Network-based Text Classification

This notebook implements a neural network-based Chinese humor detector with two tokenization strategies:

1. `basic_char`: keep only Chinese characters and treat each character as one token.
2. `advanced_jieba`: segment Chinese spans into words with `jieba`, while preserving English words, numbers, and punctuations as separate tokens.

The model uses `nn.EmbeddingBag` for bag-of-words encoding and a fully-connected classifier with two hidden layers.


### 0. Import Necessary Libraries


In [1]:
import pandas as pd
import torch

from a1_nn_utils import (
    ExperimentConfig,
    HumorClassifier,
    compare_vocab_sizes,
    dataset_summary,
    load_jsonl,
    make_train_val_split,
    preview_tokenization,
    run_all_experiments,
    seed_everything,
)

pd.set_option("display.max_colwidth", 200)

config = ExperimentConfig(
    batch_size=64,
    embed_dim=128,
    hidden_dims=(128, 64),
    dropout=0.2,
    learning_rate=2e-3,
    epochs=10,
    min_freq=2,
    val_size=0.1,
    seed=42,
    device="cpu",
)

seed_everything(config.seed)
print(config)
print("torch version:", torch.__version__)


ExperimentConfig(train_path='train.jsonl', test_path='test.jsonl', batch_size=64, embed_dim=128, hidden_dims=(128, 64), dropout=0.2, learning_rate=0.002, epochs=10, min_freq=2, val_size=0.1, seed=42, device='cpu')
torch version: 2.10.0+cpu


### 1. Data Processing


In [2]:
summary_df = dataset_summary(config.train_path, config.test_path)
summary_df


,split,num_examples,label_0,label_1,positive_ratio
0,train,12677,9031,3646,0.2876
1,test,651,481,170,0.2611


In [3]:
full_train_examples = load_jsonl(config.train_path)
train_examples, val_examples = make_train_val_split(
    full_train_examples,
    val_size=config.val_size,
    seed=config.seed,
)
example_text = train_examples[0][0]
print("Example sentence:", example_text)
preview_tokenization(example_text)


Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Zaoer\AppData\Local\Temp\jieba.cache


Example sentence: 胡学范说：你呀 别废话上吧 来来来


Loading model cost 0.988 seconds.
Prefix dict has been built successfully.


,tokenizer,tokens
0,basic_char,"[胡, 学, 范, 说, 你, 呀, 别, 废, 话, 上, 吧, 来, 来, 来]"
1,advanced_jieba,"[胡学范, 说, ：, 你, 呀, 别, 废话, 上, 吧, 来来来]"


In [4]:
vocab_df = compare_vocab_sizes(train_examples, min_freq=config.min_freq)
vocab_df


,tokenizer,min_freq,vocab_size,observed_tokens,avg_tokens_per_example
0,basic_char,2,2182,202194,17.72
1,advanced_jieba,2,5576,150779,13.22


`basic_char` creates a smaller vocabulary because each Chinese character maps to a token and non-Chinese symbols are discarded.
`advanced_jieba` keeps more detailed patterns such as words, English fragments, numbers, and punctuations, so the vocabulary becomes noticeably larger.


### 2. Build the Model


In [5]:
demo_vocab_size = int(vocab_df.loc[vocab_df["tokenizer"] == "basic_char", "vocab_size"].iloc[0])
demo_model = HumorClassifier(
    vocab_size=demo_vocab_size,
    embed_dim=config.embed_dim,
    hidden_dims=config.hidden_dims,
    dropout=config.dropout,
)
demo_model


HumorClassifier(
  (embedding): EmbeddingBag(2182, 128, mode='mean')
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=2, bias=True)
  )
)

Model design choices:

- `nn.EmbeddingBag(..., mode="mean")` implements a bag-of-words encoder efficiently.
- The classifier contains **two hidden layers**: `128 -> 64`.
- `ReLU + Dropout` are used to improve non-linearity and reduce overfitting.


### 3. Train and Evaluate


In [6]:
experiment_outputs, results_df = run_all_experiments(config=config, verbose=True)
results_df


[basic_char] epoch=01 train_loss=0.5821 val_accuracy=0.7129 val_f1=0.0162
[basic_char] epoch=02 train_loss=0.5416 val_accuracy=0.7066 val_f1=0.3086
[basic_char] epoch=03 train_loss=0.5177 val_accuracy=0.6774 val_f1=0.3195
[basic_char] epoch=04 train_loss=0.4967 val_accuracy=0.7003 val_f1=0.2245
[basic_char] epoch=05 train_loss=0.4687 val_accuracy=0.6972 val_f1=0.2351
[basic_char] epoch=06 train_loss=0.4365 val_accuracy=0.6767 val_f1=0.3408
[basic_char] epoch=07 train_loss=0.4015 val_accuracy=0.6767 val_f1=0.2679
[basic_char] epoch=08 train_loss=0.3618 val_accuracy=0.6759 val_f1=0.3295
[basic_char] epoch=09 train_loss=0.3253 val_accuracy=0.6822 val_f1=0.2712
[basic_char] epoch=10 train_loss=0.2886 val_accuracy=0.6727 val_f1=0.3444
[advanced_jieba] epoch=01 train_loss=0.5804 val_accuracy=0.7121 val_f1=0.0000
[advanced_jieba] epoch=02 train_loss=0.5065 val_accuracy=0.6853 val_f1=0.2759
[advanced_jieba] epoch=03 train_loss=0.4320 val_accuracy=0.6711 val_f1=0.2968
[advanced_jieba] epoch=04 

,tokenizer,vocab_size,accuracy,precision,recall,f1,macro_precision,macro_recall,macro_f1,loss
0,advanced_jieba,5576,0.672811,0.371257,0.364706,0.367953,0.574058,0.573205,0.573614,0.871231
1,basic_char,2182,0.671275,0.360759,0.335294,0.347561,0.565775,0.562657,0.563924,1.125160


In [7]:
for tokenizer_name, output in experiment_outputs.items():
    print(f"\nTraining history for {tokenizer_name}")
    print(
        output["history"][
            ["epoch", "train_loss", "val_accuracy", "val_precision", "val_recall", "val_f1"]
        ].to_string(index=False)
    )



Training history for basic_char
 epoch  train_loss  val_accuracy  val_precision  val_recall   val_f1
     1    0.582054      0.712934       0.600000    0.008219 0.016216
     2    0.541627      0.706625       0.479769    0.227397 0.308550
     3    0.517668      0.677445       0.406780    0.263014 0.319468
     4    0.496671      0.700315       0.440000    0.150685 0.224490
     5    0.468744      0.697161       0.430657    0.161644 0.235060
     6    0.436482      0.676656       0.412451    0.290411 0.340836
     7    0.401520      0.676656       0.384615    0.205479 0.267857
     8    0.361834      0.675868       0.407258    0.276712 0.329527
     9    0.325253      0.682177       0.398936    0.205479 0.271248
    10    0.288552      0.672713       0.406716    0.298630 0.344392

Training history for advanced_jieba
 epoch  train_loss  val_accuracy  val_precision  val_recall   val_f1
     1    0.580421      0.712145       0.000000    0.000000 0.000000
     2    0.506547      0.685331 

In [8]:
best_row = results_df.iloc[0]
print("Best tokenizer in the final test comparison:")
print(best_row)


Best tokenizer in the final test comparison:
tokenizer          advanced_jieba
vocab_size                   5576
accuracy                 0.672811
precision                0.371257
recall                   0.364706
f1                       0.367953
macro_precision          0.574058
macro_recall             0.573205
macro_f1                 0.573614
loss                     0.871231
Name: 0, dtype: object


Final note:

- Both tokenizers satisfy the assignment requirements.
- In this fixed-seed run, the advanced tokenizer achieved slightly better test-set metrics, but it also required a much larger vocabulary.
- The comparison shows the tradeoff clearly: richer tokenization can capture more patterns, but it increases model complexity.
